# Vígil.ia — YOLO11m COMPLETO (do zero) → modelo final de produção

Pipeline inteiro num só notebook, para **adotar o 11m 100%**:

1. **Estágio 1 — base 12,5k:** 11m partindo do COCO, treinado no dataset Roboflow
   (12.528 imagens, classificação → detecção via pseudo-rótulo Otsu). ~1,5–2h na A100
   (11m é ~2× o custo do 11s — mais lento por época).
2. **FT1 — fotos reais originais:** fine-tune nas fotos que fizeram o campeão
   (+ frames de vídeo de defeito), com cenas sintéticas multi-grão.
3. **FT2 — suas capturas:** fine-tune nas capturas revisadas (`soja pra treino`),
   com a **caixa apertada** (máscara limpa/erodida, borda crispa, fundo escuro realista).
4. **Vídeo:** `teste_soja.mp4` em 2 passadas (veredito final desde o 1º frame, caixas
   suavizadas) — compara o 11m final com o campeão 11n.

Cada estágio salva no Drive e **pula se já existe** (resumível). GPU: **A100** recomendada.

> Honestidade: as caixas de treino são pseudo-rótulo (Otsu) + sintético — boas, mas não
> anotação humana. O juiz final é o vídeo real; o placar sintético é só indicativo.

## 0. Setup

In [ ]:
!pip -q install "ultralytics==8.4.80"
import ultralytics; ultralytics.checks()

## 1. Config — caminhos, 11m do zero, saídas no Drive

In [ ]:
import os, glob, shutil
from google.colab import drive
drive.mount('/content/drive')

# ===== 11m DO ZERO: base 12,5k -> FT1 (fotos reais) -> FT2 (capturas) -> vídeo =====
MODEL_SCRATCH = 'yolo11m.pt'          # COCO -> treina do zero no domínio soja
SIZE = 640
BATCH = 64            # A100: pode subir p/ 128 (ou batch=-1 auto)
FRAME_STRIDE = 5
EXCLUIR_DO_TREINO = {'intact'}        # intact NUNCA treina -> avaliação limpa
# capturas (FT2) — cenas multi-grão sintéticas
N_SYNTH = 1200
N_VAL_SCENES = 120
BLUR_FRAC = 0.4
VAL_FRAC = 0.15

# --- 12,5k Roboflow (estágio 1) ---
CLS_BASE_CANDS = [
    '/content/drive/MyDrive/SoyaBeans Classifications.v2i.folder',
    '/content/drive/MyDrive/SoyaBeans Classifications.v2i.folder (Unzipped Files)',
]
CLS_BASE = next((p for p in CLS_BASE_CANDS if os.path.isdir(p)), None)
assert CLS_BASE, 'dataset 12,5k não encontrado:\n  ' + '\n  '.join(CLS_BASE_CANDS)

# --- fotos reais originais (FT1) — as que fizeram o campeão ---
REAL_SRCS = [p for p in [
    '/content/drive/MyDrive/Soja total/Soja total/Lotes',
    '/content/drive/MyDrive/Soja pra completar',
] if os.path.isdir(p)]
HAS_FT1 = len(REAL_SRCS) > 0

# --- vídeos de defeito p/ enriquecer o FT1 (opcional) ---
VAL_ROOT = '/content/drive/MyDrive/Vídeos para treino/Treino'
HAS_VAL_ROOT = os.path.isdir(VAL_ROOT)

# --- suas capturas revisadas (FT2) ---
CAP_SRCS = ['/content/drive/MyDrive/soja pra treino']
for p in CAP_SRCS:
    assert os.path.isdir(p), f'capturas não encontradas: {p}'

# saídas no Drive
STAGE1_PT = '/content/drive/MyDrive/soja_yolo11m_stage1_12k.pt'
FT1_PT    = '/content/drive/MyDrive/soja_yolo11m_ft1_reais.pt'
FINAL_PT  = '/content/drive/MyDrive/soja_yolo11m_final_v3.pt'

print('12,5k :', CLS_BASE)
print('FT1   :', REAL_SRCS if HAS_FT1 else 'PULADO (fotos reais originais não achadas)')
print('vídeos:', VAL_ROOT if HAS_VAL_ROOT else 'PULADO')
print('FT2   :', CAP_SRCS)

## 2. Funções de dataset

Idênticas ao `treino_campeao_640.ipynb` (base 12,5k, fotos reais), **com a caixa
apertada** desta versão: `extract_cutout` limpa/eroda a máscara (sem halo),
`make_scene` cola borda crispa em fundo escuro realista, `build_ft` monta as cenas
multi-grão do FT2.

In [ ]:
import glob, hashlib, unicodedata, cv2, yaml
import numpy as np

NAMES = ['broken', 'immature', 'intact', 'skin-damaged', 'spotted']
ALIASES = {0: ['broken', 'quebrad'], 1: ['immature', 'imatur', 'nao maduro'],
           2: ['intact'], 3: ['skin', 'casca', 'ardid', 'danific'], 4: ['spotted', 'manchad']}
IGNORE = ['part of the original']
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
RNG = np.random.default_rng(42)

def norm(s):
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode().lower()

def class_of(folder):
    n = norm(folder)
    if any(norm(k) in n for k in IGNORE):
        return None
    for idx in range(5):
        if any(norm(k) in n for k in ALIASES[idx]):
            return idx
    return None

def collect_real(srcs, val_frac=0.15):
    items = []
    for src in srcs:
        for root, _, files in os.walk(src):
            cls = None
            for part in reversed(root.split(os.sep)):
                c = class_of(part)
                if c is not None:
                    cls = c; break
            if cls is None:
                continue
            for fn in files:
                if fn.lower().endswith(IMG_EXT):
                    p = os.path.join(root, fn)
                    h = int(hashlib.md5(p.encode()).hexdigest(), 16)
                    items.append((p, cls, 'val' if (h % 100) < val_frac * 100 else 'train'))
    from collections import Counter
    print('coletado:', dict(Counter(sp for _, _, sp in items)))
    return items

def sat_box(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    s = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    h, w = img.shape[:2]
    if area < 0.01 * h * w or area > 0.90 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)

def otsu_box(img):
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    if area < 0.005 * h * w or area > 0.995 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)

def full_box(img):
    # fallback p/ recorte apertado do vigil_deck (grão ocupa o quadro todo):
    # a caixa É o quadro com folga mínima
    return (0.5, 0.5, 0.96, 0.96)


def letterbox640(img, size=640):
    h, w = img.shape[:2]
    s = size / max(h, w)
    img = cv2.resize(img, (max(1, round(w * s)), max(1, round(h * s))))
    h, w = img.shape[:2]
    top, left = (size - h) // 2, (size - w) // 2
    img = cv2.copyMakeBorder(img, top, size - h - top, left, size - w - left,
                             cv2.BORDER_CONSTANT, value=(0, 0, 0))
    return img, s, left, top

def motion_blur(img, rng=RNG):
    k = int(rng.choice([7, 9, 11, 13, 15]))
    kernel = np.zeros((k, k), np.float32)
    kernel[k // 2, :] = 1.0
    M = cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), float(rng.uniform(0, 180)), 1)
    kernel = cv2.warpAffine(kernel, M, (k, k))
    kernel /= max(kernel.sum(), 1e-6)
    return cv2.filter2D(img, -1, kernel)

def extract_cutout(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    s = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    h, w = img.shape[:2]
    if area < 0.01 * h * w or area > 0.90 * h * w:
        return None
    bx, by, bw, bh = cv2.boundingRect(c)
    # solidez: grão é compacto; blob esfarrapado (área << caixa) daria caixa larga -> rejeita
    if area < 0.55 * bw * bh:
        return None
    mask = np.zeros((h, w), np.uint8)
    cv2.drawContours(mask, [c], -1, 255, -1)
    # erode 1px: puxa a borda pra dentro e tira o halo -> a caixa cola no grão
    mask = cv2.erode(mask, np.ones((3, 3), np.uint8))
    ys, xs = np.where(mask > 0)
    if not len(xs):
        return None
    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1
    # recorta na bbox APERTADA da máscara já limpa
    return img[y0:y1, x0:x1], mask[y0:y1, x0:x1]


def make_bg(size, rng):
    # fundo ESCURO como o vídeo real (fundo preto/cinza), com gradiente de luz de cima + textura
    base = int(rng.integers(8, 70))
    canvas = np.full((size, size, 3), float(base), np.float32)
    grad = np.linspace(rng.uniform(-15, 5), rng.uniform(-5, 15), size)[:, None, None]
    canvas += grad
    canvas += rng.normal(0, 8, (size, size, 3))
    return np.clip(canvas, 0, 255).astype(np.uint8)


def make_scene(cutouts, rng=RNG, size=640, overlap_max=None):
    # overlap_max varia por cena: de "solto" a "encostado". Teto 0.45 (não 0.55):
    # oclusão extrema ensina caixa alucinada; aqui a prioridade é caixa que cola no grão
    if overlap_max is None:
        overlap_max = float(rng.uniform(0.10, 0.45))
    canvas = make_bg(size, rng)
    occ = np.zeros((size, size), np.uint8)
    boxes = []
    n_graos = int(rng.integers(6, 26)) if overlap_max < 0.30 else int(rng.integers(14, 36))
    for _ in range(n_graos):
        cls, crop, mask = cutouts[int(rng.integers(len(cutouts)))]
        s = int(rng.integers(60, 150)) / max(crop.shape[:2])
        crop2 = cv2.resize(crop, None, fx=s, fy=s)
        mask2 = cv2.resize(mask, None, fx=s, fy=s, interpolation=cv2.INTER_NEAREST)
        h2, w2 = crop2.shape[:2]
        diag = int(np.ceil(np.hypot(h2, w2))) + 2
        M = cv2.getRotationMatrix2D((w2 / 2, h2 / 2), float(rng.uniform(0, 360)), 1)
        M[0, 2] += (diag - w2) / 2
        M[1, 2] += (diag - h2) / 2
        crop3 = cv2.warpAffine(crop2, M, (diag, diag))
        mask3 = cv2.warpAffine(mask2, M, (diag, diag), flags=cv2.INTER_NEAREST)
        ys, xs = np.where(mask3 > 0)
        if not len(xs):
            continue
        crop3 = crop3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        mask3 = mask3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        gh, gw = mask3.shape
        if gh >= size - 2 or gw >= size - 2:
            continue
        placed = False
        for _try in range(25):
            px = int(rng.integers(0, size - gw))
            py = int(rng.integers(0, size - gh))
            inter = (occ[py:py + gh, px:px + gw] > 0) & (mask3 > 0)
            if inter.sum() <= overlap_max * (mask3 > 0).sum():
                placed = True
                break
        if not placed:
            continue
        # sombreamento leve por grão: não fica com cara de "colado", ajuda a separar do fundo
        shade = float(rng.uniform(0.75, 1.1))
        crop3 = np.clip(crop3.astype(np.float32) * shade, 0, 255).astype(np.uint8)
        # feather MENOR (3,3): borda crispa -> o grão visível casa com a caixa, sem halo
        alpha = (cv2.GaussianBlur(mask3, (3, 3), 0).astype(np.float32) / 255)[..., None]
        reg = canvas[py:py + gh, px:px + gw]
        canvas[py:py + gh, px:px + gw] = (alpha * crop3 + (1 - alpha) * reg).astype(np.uint8)
        occ[py:py + gh, px:px + gw][mask3 > 0] = 255
        boxes.append((cls, (px + gw / 2) / size, (py + gh / 2) / size, gw / size, gh / size))
    return canvas, boxes


def balance_train(items):
    from collections import defaultdict
    train = [it for it in items if it[2] == 'train']
    rest = [it for it in items if it[2] != 'train']
    by = defaultdict(list)
    for it in train:
        by[it[1]].append(it)
    mx = max(len(v) for v in by.values())
    out = []
    for c, v in by.items():
        out += v + [v[int(i)] for i in RNG.integers(0, len(v), mx - len(v))]
    print('balanceado (train):', {NAMES[c]: sum(1 for it in out if it[1] == c) for c in sorted(by)})
    return out + rest

def build_v3(items, out_dir, n_synth=600, blur_frac=0.4):
    assert items, 'Nenhuma imagem coletada! Confira REAL_SRCS.'
    for sp in ('train', 'val', 'test'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)
    items = balance_train(items)
    cutouts = []
    kept = skipped = 0
    for i, (path, cls, sp) in enumerate(items):
        if i % 200 == 0:
            print(f'  fotos {i}/{len(items)}…', flush=True)
        img = cv2.imread(path)
        if img is None:
            skipped += 1; continue
        h0, w0 = img.shape[:2]
        box = sat_box(img) or otsu_box(img)
        if box is None:
            skipped += 1; continue
        if sp == 'train':
            cut = extract_cutout(img)
            if cut is not None:
                cutouts.append((cls, cut[0], cut[1]))
        lb, s, left, top = letterbox640(img)
        cx, cy, ww, hh = box
        cx = (cx * w0 * s + left) / 640.0
        cy = (cy * h0 * s + top) / 640.0
        ww = (ww * w0 * s) / 640.0
        hh = (hh * h0 * s) / 640.0
        line = f'{cls} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}'
        stem = f'{sp}_{i:06d}'
        cv2.imwrite(f'{out_dir}/images/{sp}/{stem}.jpg', lb, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/{sp}/{stem}.txt', 'w').write(line)
        kept += 1
        if sp == 'train':
            cv2.imwrite(f'{out_dir}/images/train/{stem}b.jpg', motion_blur(lb),
                        [cv2.IMWRITE_JPEG_QUALITY, 95])
            open(f'{out_dir}/labels/train/{stem}b.txt', 'w').write(line)
            kept += 1
    print(f'fotos reais: kept={kept} skipped={skipped} | recortes: {len(cutouts)}')
    assert cutouts, 'Nenhum recorte extraído!'
    synth = 0
    for j in range(n_synth):
        if j % 100 == 0:
            print(f'  cenas {j}/{n_synth}…', flush=True)
        canvas, boxes = make_scene(cutouts)
        if not boxes:
            continue
        if RNG.random() < blur_frac:
            canvas = motion_blur(canvas)
        stem = f'synth_{j:05d}'
        cv2.imwrite(f'{out_dir}/images/train/{stem}.jpg', canvas, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/train/{stem}.txt', 'w').write(
            '\n'.join(f'{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}' for c, cx, cy, w, h in boxes))
        synth += 1
    print(f'cenas sintéticas: {synth}')
    yaml.safe_dump({'path': out_dir, 'train': 'images/train', 'val': 'images/val',
                    'test': 'images/test', 'names': {i: n for i, n in enumerate(NAMES)}},
                   open(f'{out_dir}/data.yaml', 'w'), sort_keys=False, allow_unicode=True)
    return f'{out_dir}/data.yaml'

SPLIT_MAP = {'train': 'train', 'valid': 'val', 'val': 'val', 'test': 'test'}

def collect_base(base_dir):
    """Acha train/valid/test em qualquer profundidade dentro do dataset 12,5k."""
    items = []
    for root, dirs, _ in os.walk(base_dir):
        for d in list(dirs):
            sp = SPLIT_MAP.get(d.lower())
            if sp is None:
                continue
            split_dir = os.path.join(root, d)
            for folder in sorted(os.listdir(split_dir)):
                cls = class_of(folder)
                if cls is None:
                    continue
                for p_ in glob.glob(os.path.join(split_dir, folder, '*')):
                    if p_.lower().endswith(IMG_EXT):
                        items.append((p_, cls, sp))
            dirs.remove(d)
    from collections import Counter
    print('coletado (base 12,5k):', dict(Counter(sp for _, _, sp in items)))
    return items

def build_base(items, out_dir):
    """Dataset base de detecção: pseudo-rótulo Otsu (1 grão/img, fundo preto),
    sem balanceamento/blur/sintético — idêntico ao estágio base do RT-DETR."""
    assert items, 'Nenhuma imagem do 12,5k coletada!'
    for sp in ('train', 'val', 'test'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)
    kept = skipped = 0
    for i, (path, cls, sp) in enumerate(items):
        if i % 1000 == 0:
            print(f'  base {i}/{len(items)}…', flush=True)
        img = cv2.imread(path)
        if img is None:
            skipped += 1; continue
        h0, w0 = img.shape[:2]
        box = otsu_box(img)
        if box is None:
            skipped += 1; continue
        lb, s, left, top = letterbox640(img)
        cx, cy, ww, hh = box
        cx = (cx * w0 * s + left) / 640.0
        cy = (cy * h0 * s + top) / 640.0
        ww = (ww * w0 * s) / 640.0
        hh = (hh * h0 * s) / 640.0
        stem = f'{sp}_{i:06d}'
        cv2.imwrite(f'{out_dir}/images/{sp}/{stem}.jpg', lb, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/{sp}/{stem}.txt', 'w').write(
            f'{cls} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}')
        kept += 1
    print(f'base: kept={kept} skipped={skipped}')
    yaml.safe_dump({'path': out_dir, 'train': 'images/train', 'val': 'images/val',
                    'test': 'images/test', 'names': {i: n for i, n in enumerate(NAMES)}},
                   open(f'{out_dir}/data.yaml', 'w'), sort_keys=False, allow_unicode=True)
    return f'{out_dir}/data.yaml'

# --- correção #2: escala do grão proporcional ao canvas (identidade em 640,
#     mas mantém o pipeline consistente com o notebook 1280) ---
def make_scene(cutouts, rng=RNG, size=640):
    bg = int(rng.integers(20, 130))
    canvas = np.clip(np.full((size, size, 3), bg, np.int16)
                     + rng.normal(0, 6, (size, size, 3)), 0, 255).astype(np.uint8)
    occ = np.zeros((size, size), np.uint8)
    boxes = []
    lo, hi = int(60 * size / 640), int(150 * size / 640)
    for _ in range(int(rng.integers(6, 26))):
        cls, crop, mask = cutouts[int(rng.integers(len(cutouts)))]
        s = int(rng.integers(lo, hi)) / max(crop.shape[:2])
        crop2 = cv2.resize(crop, None, fx=s, fy=s)
        mask2 = cv2.resize(mask, None, fx=s, fy=s, interpolation=cv2.INTER_NEAREST)
        h2, w2 = crop2.shape[:2]
        diag = int(np.ceil(np.hypot(h2, w2))) + 2
        M = cv2.getRotationMatrix2D((w2 / 2, h2 / 2), float(rng.uniform(0, 360)), 1)
        M[0, 2] += (diag - w2) / 2
        M[1, 2] += (diag - h2) / 2
        crop3 = cv2.warpAffine(crop2, M, (diag, diag))
        mask3 = cv2.warpAffine(mask2, M, (diag, diag), flags=cv2.INTER_NEAREST)
        ys, xs = np.where(mask3 > 0)
        if not len(xs):
            continue
        crop3 = crop3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        mask3 = mask3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        gh, gw = mask3.shape
        if gh >= size - 2 or gw >= size - 2:
            continue
        placed = False
        for _try in range(20):
            px = int(rng.integers(0, size - gw))
            py = int(rng.integers(0, size - gh))
            inter = (occ[py:py + gh, px:px + gw] > 0) & (mask3 > 0)
            if inter.sum() <= 0.15 * (mask3 > 0).sum():
                placed = True
                break
        if not placed:
            continue
        alpha = (cv2.GaussianBlur(mask3, (5, 5), 0).astype(np.float32) / 255)[..., None]
        reg = canvas[py:py + gh, px:px + gw]
        canvas[py:py + gh, px:px + gw] = (alpha * crop3 + (1 - alpha) * reg).astype(np.uint8)
        occ[py:py + gh, px:px + gw][mask3 > 0] = 255
        boxes.append((cls, (px + gw / 2) / size, (py + gh / 2) / size, gw / size, gh / size))
    return canvas, boxes

print('funções prontas (make_scene proporcional)')


def build_ft(items, out_dir, n_synth=N_SYNTH, n_val_scenes=N_VAL_SCENES,
             blur_frac=BLUR_FRAC):
    assert items, 'Nenhuma imagem coletada! Confira REAL_SRCS.'
    import shutil
    shutil.rmtree(out_dir, ignore_errors=True)
    for sp in ('train', 'val'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)
    items = balance_train(items)

    cut_train, cut_val = [], []
    singles = skipped = 0
    for i, (path, cls, sp) in enumerate(items):
        if i % 200 == 0:
            print(f'  fotos {i}/{len(items)}…', flush=True)
        img = cv2.imread(path)
        if img is None:
            skipped += 1; continue
        cut = extract_cutout(img)
        if cut is not None:
            (cut_train if sp == 'train' else cut_val).append((cls, cut[0], cut[1]))
        # recorte solto só vira IMAGEM de treino se tiver caixa real (não o quadro
        # inteiro) — senão ensina o vício "caixa = quadro" que já derrubou o base_12k
        if sp != 'train':
            continue
        box = sat_box(img) or otsu_box(img)
        if box is None or box[2] * box[3] > 0.75:
            continue
        h0, w0 = img.shape[:2]
        lb, s, left, top = letterbox640(img)
        cx, cy, ww, hh = box
        cx = (cx * w0 * s + left) / 640.0
        cy = (cy * h0 * s + top) / 640.0
        ww = (ww * w0 * s) / 640.0
        hh = (hh * h0 * s) / 640.0
        line = f'{cls} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}'
        stem = f'single_{i:06d}'
        cv2.imwrite(f'{out_dir}/images/train/{stem}.jpg', lb, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/train/{stem}.txt', 'w').write(line)
        cv2.imwrite(f'{out_dir}/images/train/{stem}b.jpg', motion_blur(lb),
                    [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/train/{stem}b.txt', 'w').write(line)
        singles += 2
    print(f'singles c/ caixa real: {singles} | skipped: {skipped} | '
          f'recortes train: {len(cut_train)} | recortes val: {len(cut_val)}')
    assert cut_train, 'Nenhum recorte de treino extraído!'
    assert cut_val, ('Nenhum recorte de VAL extraído — sem como montar o val '
                     'multi-grão. Confira as imagens do split val.')

    def write_scenes(cutouts, split, n, rng, blur):
        made = 0
        for j in range(n):
            if j % 200 == 0:
                print(f'  cenas {split} {j}/{n}…', flush=True)
            canvas, boxes = make_scene(cutouts, rng=rng)
            if not boxes:
                continue
            if rng.random() < blur:
                canvas = motion_blur(canvas, rng=rng)
            stem = f'synth_{split}_{j:05d}'
            cv2.imwrite(f'{out_dir}/images/{split}/{stem}.jpg', canvas,
                        [cv2.IMWRITE_JPEG_QUALITY, 95])
            open(f'{out_dir}/labels/{split}/{stem}.txt', 'w').write(
                '\n'.join(f'{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}'
                           for c, cx, cy, w, h in boxes))
            made += 1
        print(f'cenas {split}: {made}')

    write_scenes(cut_train, 'train', n_synth, np.random.default_rng(42), blur_frac)
    write_scenes(cut_val, 'val', n_val_scenes, np.random.default_rng(123), blur_frac)

## 3. Constrói base 12,5k (estágio 1) + fotos reais (FT1)

In [ ]:
import yaml
# ---- estágio 1: base 12,5k (pseudo-rótulo Otsu) ----
BASE_YAML = '/content/soja_det_base/data.yaml'
if not os.path.exists(BASE_YAML):
    print('construindo base 12,5k (~10 min)…')
    BASE_YAML = build_base(collect_base(CLS_BASE), '/content/soja_det_base')
print('base 12,5k:', BASE_YAML)

# ---- FT1: fotos reais originais (v3: real + blur + cenas sintéticas) ----
V3_YAML = None
if HAS_FT1:
    V3_YAML = '/content/soja_det_v3/data.yaml'
    if not os.path.exists(V3_YAML):
        print('construindo v3 fotos reais (~10-15 min)…')
        build_v3(collect_real(REAL_SRCS), '/content/soja_det_v3')
        yaml.safe_dump({'train': '/content/soja_det_v3/images/train',
                        'val': '/content/soja_det_v3/images/val',
                        'names': {i: n for i, n in enumerate(NAMES)}},
                       open(V3_YAML, 'w'), sort_keys=False, allow_unicode=True)
    print('v3 fotos reais:', V3_YAML)
else:
    print('FT1 pulado — sem fotos reais originais no Drive')

### 3b. Frames de vídeo de defeito para o FT1 (opcional)

In [ ]:
if not HAS_VAL_ROOT:
    HAS_VIDEO_DATA = False
    print('vídeos de defeito: PULADO (VAL_ROOT ausente)')
else:
    VIDEO_EXT = ('.mp4', '.mov', '.avi', '.mkv')
    ROI_BOTTOM = 0.15
    MIN_SAT    = 40

    def video_frames(path, stride=FRAME_STRIDE):
        cap = cv2.VideoCapture(path)
        k = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if k % stride == 0:
                yield k, frame[:int(frame.shape[0] * (1 - ROI_BOTTOM))]
            k += 1
        cap.release()

    def _dedupe(boxes):
        out = []
        boxes = sorted(boxes, key=lambda b: -(b[2] * b[3]))
        for b in boxes:
            bx1, by1, bx2, by2 = b[0]-b[2]/2, b[1]-b[3]/2, b[0]+b[2]/2, b[1]+b[3]/2
            dup = False
            for k in out:
                kx1, ky1, kx2, ky2 = k[0]-k[2]/2, k[1]-k[3]/2, k[0]+k[2]/2, k[1]+k[3]/2
                iw = max(0, min(bx2, kx2) - max(bx1, kx1))
                ih = max(0, min(by2, ky2) - max(by1, ky1))
                if iw * ih > 0.4 * (b[2] * b[3]):
                    dup = True; break
            if not dup:
                out.append(b)
        return out

    def multi_boxes(img, min_frac=0.0015, max_frac=0.05, edge=0.02):
        h, w = img.shape[:2]
        S = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)[:, :, 1]
        blur = cv2.GaussianBlur(S, (5, 5), 0)
        _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
        cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        boxes = []
        for c in cnts:
            a = cv2.contourArea(c)
            if not (min_frac * h * w <= a <= max_frac * h * w):
                continue
            x, y, bw, bh = cv2.boundingRect(c)
            if bw / max(bh, 1) > 2.2 or bh / max(bw, 1) > 2.2:
                continue
            if a < 0.5 * bw * bh:
                continue
            if S[y:y+bh, x:x+bw].mean() < MIN_SAT:
                continue
            cx, cy = (x + bw/2) / w, (y + bh/2) / h
            if not (edge < cx < 1-edge and edge < cy < 1-edge):
                continue
            pad = int(0.04 * min(bw, bh)) + 2
            x1, y1 = max(0, x - pad), max(0, y - pad)
            x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
            boxes.append((((x1+x2)/2)/w, ((y1+y2)/2)/h, (x2-x1)/w, (y2-y1)/h))
        return _dedupe(boxes)

    VID = '/content/soja_video_640'
    import shutil as _sh
    _sh.rmtree(VID, ignore_errors=True)
    for sp in ('train', 'val'):
        os.makedirs(f'{VID}/images/{sp}', exist_ok=True)
        os.makedirs(f'{VID}/labels/{sp}', exist_ok=True)

    stats = {}
    for entry in sorted(os.listdir(VAL_ROOT)):
        sub = os.path.join(VAL_ROOT, entry)
        if not os.path.isdir(sub):
            continue
        cls = class_of(entry)
        if cls is None:
            continue
        if NAMES[cls] in EXCLUIR_DO_TREINO:
            print(f'{entry} -> {NAMES[cls]}: EXCLUÍDO do treino (avaliação limpa)')
            continue
        vids = [os.path.join(sub, f) for f in sorted(os.listdir(sub))
                if f.lower().endswith(VIDEO_EXT)]
        if not vids:
            continue
        val_video = vids[-1] if len(vids) >= 2 else None
        wrote = {'train': 0, 'val': 0}
        for vi, vp in enumerate(vids):
            frames = list(video_frames(vp))
            if val_video is None:
                cut = int(0.85 * len(frames))
                split_of = lambda i: ('train' if i < cut else
                                      None if i < cut + 150 // FRAME_STRIDE else 'val')
            else:
                split_of = lambda i: 'val' if vp == val_video else 'train'
            for i, (k, frame) in enumerate(frames):
                sp = split_of(i)
                if sp is None:
                    continue
                boxes = multi_boxes(frame)
                if not (3 <= len(boxes) <= 80):
                    continue
                lb, s, left, top = letterbox640(frame, size=SIZE)
                H0, W0 = frame.shape[:2]
                lines = []
                for cx, cy, ww, hh in boxes:
                    cx2 = (cx * W0 * s + left) / SIZE
                    cy2 = (cy * H0 * s + top) / SIZE
                    lines.append(f'{cls} {cx2:.6f} {cy2:.6f} {ww*W0*s/SIZE:.6f} {hh*H0*s/SIZE:.6f}')
                stem = f'{NAMES[cls]}_{vi}_{k:05d}'
                cv2.imwrite(f'{VID}/images/{sp}/{stem}.jpg', lb, [cv2.IMWRITE_JPEG_QUALITY, 95])
                open(f'{VID}/labels/{sp}/{stem}.txt', 'w').write('\n'.join(lines))
                wrote[sp] += 1
        stats[NAMES[cls]] = wrote
        print(f'{entry} -> {NAMES[cls]}: {wrote}')

    HAS_VIDEO_DATA = any(v['train'] for v in stats.values())
    print('\nvídeo no fine-tune:', stats if HAS_VIDEO_DATA else
          'NENHUM (só intact existe e está excluído) — estágio 2 será só fotos reais')

## 4. Constrói o dataset das SUAS capturas (FT2, caixa apertada)

In [ ]:
CAP_OUT = '/content/soja_cap'
build_ft(collect_real(CAP_SRCS), CAP_OUT, n_synth=N_SYNTH,
         n_val_scenes=N_VAL_SCENES, blur_frac=BLUR_FRAC)
CAP_YAML = f'{CAP_OUT}/data.yaml'
yaml.safe_dump({'train': f'{CAP_OUT}/images/train', 'val': f'{CAP_OUT}/images/val',
                'names': {i: n for i, n in enumerate(NAMES)}},
               open(CAP_YAML, 'w'), sort_keys=False, allow_unicode=True)
print('capturas:', CAP_YAML)

## 5. Treino em 3 estágios (11m do zero → final)

In [ ]:
import shutil
from ultralytics import YOLO

COMMON = dict(
    imgsz=SIZE, device=0, seed=42, optimizer='AdamW',
    cache=False, workers=8,
    mosaic=1.0, hsv_v=0.5, degrees=15, translate=0.1, scale=0.5,
    fliplr=0.5, flipud=0.5,
    project='runs_11m', exist_ok=True,
)

# ===== ESTÁGIO 1: 11m do zero (COCO) na base 12,5k =====
if os.path.exists(STAGE1_PT):
    print('estágio 1 já no Drive:', STAGE1_PT)
else:
    print('ESTÁGIO 1: 11m <- COCO na base 12,5k (~1-1,5h na A100)')
    m1 = YOLO(MODEL_SCRATCH)
    m1.train(name='11m_stage1_12k', data=BASE_YAML, batch=BATCH, epochs=50,
             lr0=0.001, patience=20, close_mosaic=10, **COMMON)
    shutil.copy(str(m1.trainer.best), STAGE1_PT)
    print('salvo:', STAGE1_PT)

# ===== FT1: fine-tune nas fotos reais originais (+ vídeos de defeito) =====
if not HAS_FT1:
    FT1_PT = STAGE1_PT
    print('FT1 pulado — usando o estágio 1 direto como base do FT2')
elif os.path.exists(FT1_PT):
    print('FT1 já no Drive:', FT1_PT)
else:
    MIX_YAML = '/content/soja_mix_ft1.yaml'
    train_dirs = ['/content/soja_det_v3/images/train']
    if HAS_VIDEO_DATA:
        train_dirs.append('/content/soja_video_640/images/train')
    yaml.safe_dump({'train': train_dirs, 'val': '/content/soja_det_v3/images/val',
                    'names': {i: n for i, n in enumerate(NAMES)}},
                   open(MIX_YAML, 'w'), sort_keys=False, allow_unicode=True)
    print('FT1: fine-tune fotos reais ->', train_dirs)
    m2 = YOLO(STAGE1_PT)
    m2.train(name='11m_ft1_reais', data=MIX_YAML, batch=BATCH, epochs=60,
             lr0=0.001, patience=20, close_mosaic=8, **COMMON)
    shutil.copy(str(m2.trainer.best), FT1_PT)
    print('salvo:', FT1_PT)

# ===== FT2: fine-tune nas SUAS capturas (caixa apertada, multi-grão) =====
if os.path.exists(FINAL_PT):
    print('FT2 (final) já no Drive:', FINAL_PT)
else:
    print('FT2: fine-tune capturas ->', FINAL_PT)
    m3 = YOLO(FT1_PT)
    m3.train(name='11m_ft2_capturas', data=CAP_YAML, batch=BATCH, epochs=60,
             lr0=0.0005, patience=20, close_mosaic=8, **COMMON)
    shutil.copy(str(m3.trainer.best), FINAL_PT)
    print('MODELO FINAL salvo:', FINAL_PT)

## 6. Juiz de verdade — `teste_soja.mp4` (11m final vs campeão 11n)

2 passadas (votos → render com veredito final desde o 1º frame), caixas suavizadas,
mesma regra exigente do `vigil_deck.py`. Gera 2 vídeos no Drive.

In [ ]:
import os
from collections import defaultdict, Counter
import cv2
from ultralytics import YOLO

def first_existing(*paths):
    return next((p for p in paths if os.path.exists(p)), None)

VIDEO_TESTE = first_existing('/content/drive/MyDrive/teste_soja.mp4',
                             '/content/drive/MyDrive/teste_soja.avi')
assert VIDEO_TESTE, 'teste_soja.(mp4|avi) não encontrado no Drive!'

# regra exigente por classe (igual ao vigil_deck.py)
RATIOS = {'broken': 0.85, 'skin-damaged': 0.80, 'spotted': 0.75, 'immature': 0.75}
MIN_TRACK_FRAMES = 3    # grão visto em menos frames que isso = ruído (não desenha)
SMOOTH = 0.4            # suavização da caixa (EMA): menor = mais estável, menos treme
CONF = 0.30             # baixe p/ 0.25 se faltar caixa; suba p/ 0.4 se aparecer lixo
PT_LABEL = {'broken': 'Quebrado', 'immature': 'Imaturo', 'intact': 'Intacto',
            'skin-damaged': 'Casca danif.', 'spotted': 'Manchado'}
COLORS = {'intact': (90, 200, 90), 'immature': (60, 200, 200), 'broken': (170, 100, 210),
          'skin-damaged': (255, 160, 60), 'spotted': (70, 70, 235)}

def veredito(cnt):
    top, w = cnt.most_common(1)[0]
    if top == 'intact':
        return 'intact'
    return top if w >= RATIOS[top] * sum(cnt.values()) else 'intact'

def coletar(pt, video_path, imgsz=SIZE, conf=CONF):
    """Passada 1: junta os votos de cada grão no vídeo TODO + guarda dets por frame."""
    model = YOLO(pt)
    votes = defaultdict(Counter)
    seen = Counter()
    dets_per_frame = defaultdict(list)
    k = 0
    for r in model.track(source=video_path, imgsz=imgsz, conf=conf, iou=0.5,
                         agnostic_nms=True, tracker='bytetrack.yaml',
                         persist=True, stream=True, verbose=False):
        if r.boxes.id is not None:
            for xyxy, tid, c, cf in zip(r.boxes.xyxy.cpu().numpy().astype(int),
                                        r.boxes.id.int().tolist(),
                                        r.boxes.cls.int().tolist(),
                                        r.boxes.conf.tolist()):
                votes[tid][NAMES[c]] += cf
                seen[tid] += 1
                x1, y1, x2, y2 = xyxy
                dets_per_frame[k].append((tid, x1, y1, x2, y2))
        k += 1
    # veredito final por grão — sobre o vídeo todo (sem hold, sem "analisando")
    verdict = {tid: veredito(v) for tid, v in votes.items()
               if seen[tid] >= MIN_TRACK_FRAMES}
    return dets_per_frame, verdict, seen, k

def render(video_path, dets_per_frame, verdict, out_path):
    """Passada 2: desenha cada grão já com o veredito final, caixa suavizada."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    smooth = {}
    writer, k = None, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if writer is None:
            h, w = frame.shape[:2]
            writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
        for tid, x1, y1, x2, y2 in dets_per_frame.get(k, []):
            if tid not in verdict:      # track curto demais (ruído) -> não desenha
                continue
            if tid in smooth:           # suaviza a caixa (não treme frame a frame)
                px1, py1, px2, py2 = smooth[tid]
                x1 = int(SMOOTH * x1 + (1 - SMOOTH) * px1)
                y1 = int(SMOOTH * y1 + (1 - SMOOTH) * py1)
                x2 = int(SMOOTH * x2 + (1 - SMOOTH) * px2)
                y2 = int(SMOOTH * y2 + (1 - SMOOTH) * py2)
            smooth[tid] = (x1, y1, x2, y2)
            cls = verdict[tid]
            color = COLORS[cls]
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f'#{tid} {PT_LABEL[cls]}', (x1, max(18, y1 - 6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        writer.write(frame)
        k += 1
    cap.release(); writer.release()

# compara o 11m FINAL (do zero) com o campeão 11n implantado, se existir
REF_11N = first_existing('/content/drive/MyDrive/soja_yolo11n_base12k_v2.pt')
MODELOS_TESTE = {'novo_11m': FINAL_PT}
if REF_11N:
    MODELOS_TESTE = {'campeao_11n': REF_11N, 'novo_11m': FINAL_PT}
resultados = {}
for tag, pt in MODELOS_TESTE.items():
    print(f'>>> {tag}: passada 1 (votos) em {os.path.basename(VIDEO_TESTE)}…')
    dets, verdict, seen, n_frames = coletar(pt, VIDEO_TESTE)
    print('    passada 2 (render)…')
    out_path = f'/content/drive/MyDrive/comparativo_{tag}.mp4'
    render(VIDEO_TESTE, dets, verdict, out_path)
    dist = Counter(verdict.values())
    resultados[tag] = dist
    print(f'    {n_frames} frames | {sum(dist.values())} grãos c/ veredito | {dict(dist)}')
    print(f'    salvo: {out_path}')

print()
print('=== resumo lado a lado ===')
for cls in NAMES:
    a = resultados.get('campeao_11n', {}).get(cls, 0)
    b = resultados['novo_11m'].get(cls, 0)
    print(f'  {PT_LABEL[cls]:14s} campeão_11n={a:3d}  novo_11m={b:3d}')
print()
print('Baixe comparativo_campeao_v2.mp4 e comparativo_novo_v3.mp4 do Drive e')
print('assista lado a lado: menos caixa piscando/classe trocando = melhor.')

## 7. Export OpenVINO (iGPU Intel no Windows)

In [ ]:
# export OpenVINO FP16 do modelo FINAL (p/ a iGPU Intel no Windows)
!yolo export model={FINAL_PT} format=openvino imgsz=640 half=True
import shutil
ov = FINAL_PT.replace('.pt', '_openvino_model')
shutil.make_archive(ov, 'zip', ov)
print('zip OpenVINO:', ov + '.zip')
print('use no vigil_deck.py:  --model soja_yolo11m_final_v3_openvino_model --imgsz 640 --device intel:gpu')

## Depois

- Modelo final: **`soja_yolo11m_final_v3.pt`** no Drive (+ pasta OpenVINO zipada).
- Rode no `vigil_deck.py`:
  `--model soja_yolo11m_final_v3_openvino_model --imgsz 640 --device intel:gpu`
- Compare os 2 vídeos (`comparativo_*.mp4`): menos caixa piscando/classe trocando = melhor.
- Se o 11m ganhar do 11n no vídeo, é o novo modelo de produção. Custo: ~3× o FLOPs
  do 11n (menos fps na iGPU) — a troca vale se a qualidade de caixa compensar.